# Graph models with scarce mule labels on Kaggle GPU

Label-scarcity runs of the research plan (section 19) for the graph models: the TGN with features (`tgn_features.yaml`), the role-transition step (`tgn_transition.yaml`, both `earlymule.models.tgn`) and the STA-GT-inspired model (`stagt.yaml`, `earlymule.models.stagt`), trained with the mule labels of only a share of the training cases (`--label-fraction`), chosen by seed from the case map `label_cases_80K-earlymule-mule.csv` exactly as for LightGBM. The other training mules count as negatives; validation and test labels are untouched. A stage with fraction `None` is a full-label run.

The stages of a batch run one after another; the seeds of a stage train in parallel on the two T4 GPUs (GPU = seed % 2), two runs per GPU for the TGN and one for STA-GT, which needs about 7 GB. A stage can name its own seeds as a fourth item, to rerun failed seeds. Outputs go to `/kaggle/working/<config name>/<lf<F> or full>/seed<N>/` with a log next to each; evaluation runs locally with `python -m earlymule.evaluation.import_scores`.

Inputs: the private datasets `verzol/earlymule-80k-earlymule-mule` (export) and `verzol/earlymule-80k-label-cases` (case map). One notebook per batch is generated by `notebooks/kaggle/label_fraction/make_gpu_batches.py`, which rewrites the parameter cell.


In [ ]:
# Run parameters: stages (module, config, label fraction or None), seeds, commit to check out
STAGES = [('earlymule.models.tgn', 'tgn_transition.yaml', 0.01)]
SEEDS = [0, 1, 2]
COMMIT = None

In [ ]:
# GPU and driver
!nvidia-smi

In [ ]:
# Clone the repository at COMMIT and install it with PyG (torch is preinstalled on Kaggle)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!pip install -q torch_geometric

In [ ]:
# Run every stage in order (optional 4th item: its own seeds); a failed seed is logged and the next stage still starts
import json, os, subprocess, time
from pathlib import Path
import pyarrow.parquet as pq
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == '80K-earlymule-mule')
# Guard: the mounted export must be the 72-feature version the full-label runs used
n_feat = len(pq.read_schema(data / 'features_train.parquet').names) - 2
assert n_feat == 72, f'dataset has {n_feat} features'
label_cases = next(Path('/kaggle/input').rglob('label_cases_80K-earlymule-mule.csv'))
codes = []
for module, config, fraction, *seeds in STAGES:
    seeds = seeds[0] if seeds else SEEDS
    # Two STA-GT runs do not fit on one 15 GB T4, so STA-GT gets one run per GPU at a time
    per_gpu = 1 if module.endswith('stagt') else 2
    out = Path('/kaggle/working') / Path(config).stem / (f'lf{fraction:g}' if fraction else 'full')
    out.mkdir(parents=True, exist_ok=True)
    t0, procs = time.time(), []
    for seed in seeds:
        while sum(p.poll() is None and p.gpu == seed % 2 for p in procs) >= per_gpu:
            time.sleep(30)
        cmd = ['python', '-m', module, '--data', str(data), '--config', config, '--seed', str(seed),
               '--out', str(out / f'seed{seed}')]
        if fraction:
            cmd += ['--label-fraction', str(fraction), '--label-cases', str(label_cases)]
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(seed % 2)}
        procs.append(subprocess.Popen(cmd, cwd='/tmp/earlymule-tg', env=env,
                                      stdout=open(out / f'seed{seed}.log', 'w'), stderr=subprocess.STDOUT))
        procs[-1].gpu = seed % 2
    codes.append((config, fraction, [p.wait() for p in procs], round((time.time() - t0) / 3600, 2)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last lines of every run log
for log in sorted(Path('/kaggle/working').glob('*/*/seed*.log')):
    print(f'--- {log.parent.parent.name}/{log.parent.name}/{log.stem}')
    print(log.read_text()[-2000:])